In [ ]:
#Initial Version 

import cv2
import numpy as np
from tracker import EuclideanDistTracker
import math
import matplotlib.pyplot as plt
import csv

#Data for the Velocity
fps = 5000
frame_width_px = 640  
frame_height_py= 480

real_width_A = 52.484                                 # In cm
real_height_A = 39.363   
real_width_B =  53.5977
real_height_B = 40.198

meters_per_pixel_x = (real_width_B/100)/ frame_width_px
meters_per_pixel_y = (real_height_B/100)/frame_height_py
centroids = []
frame_inidices=[]
tracker= EuclideanDistTracker()                                         #Create tracker object( file in the same location as the py file)

video= cv2.VideoCapture(r"E:\Python Code\Boulder Detection Project\Data\Merengues-bowl_195bar_delrin_qs_vert.avi")              #Detection of the objects 
object_d= cv2.createBackgroundSubtractorKNN(1000,400, False)
#Start with the video play and looping
c=0
frame_count=0
fourcc=cv2.VideoWriter_fourcc(*'mp4v')
out=cv2.VideoWriter(r"E:\Python Code\Boulder Detection Project\Output\Video_calibration.avi",fourcc,30,(640,480),True)
while video.isOpened():
    r, frame= video.read()
    if r==True:
        filename =r"E:\Python Code\Boulder Detection Project\Output\Images\Frame"+str(c)+".png"
        h,w, _= frame.shape
        # print(h,w)
        roi= frame[25:250,150:400]                #ROI for the projectile 
        # roi_1= frame[100:300,160:300]                #RoI for the fragments/ejecta
        # mask= object_d.apply(frame)
        mask= object_d.apply(roi)
        # mask= object_d.apply(roi_1)
        # mask=object_d.apply(frame)
        _, mask= cv2.threshold(mask, 254, 255, cv2.THRESH_BINARY)                                           
        contours, _= cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_NONE)
        detection=[]
        for cnt in contours:
            area= cv2.contourArea(cnt)
            # print(area)                                                              #To determine the area of the countour that helps in the area selection for the oebject detection
            if area>350:              
                cv2.drawContours(roi, [cnt], -1, (0,255,0), 1)
                # cv2.drawContours(roi_1, [cnt], -1, (0,255,0), 1)
                # cv2.drawContours(frame,[cnt],-1,(0,255,0),1)
                x,y,w,h= cv2.boundingRect(cnt)
                # cv2.rectangle(roi, (x,y), (x+w,y+h), (0,255,0), 1)
                # cv2.rectangle(roi_1, (x,y), (x+w,y+h), (0,255,0), 1)
                # cv2.rectangle(frame, (x,y), (x+w,y+h), (0,255,0), 1)
                #cv2.circle(roi, (((x+w/2)), ((y+h)/2)), h/2, (255,0,0),1)             #To work on this.(optional)
                detection.append([x,y,w,h])
                #print(detection)
                # print(x,y,w,h) 
#Tracking of each debrie that is ejected out.(optional to use)
                boxes_ids= tracker.update(detection)
                for box_id in boxes_ids:
                    x,y,w,h,id= box_id
                    # if id==target_id:
                    cv2.rectangle(roi,(x,y), (x+w,y+h), (0,0,255), 1)
                    #cv2.putText(roi_1, str(id), (x,y-15), cv2.FONT_HERSHEY_PLAIN, 1, (255,0,0))
                    # print(x,y,w,h,id)
#For the calculation of velocity, we direct the detection values to the pixel points and find the location:
            if detection:
                    x,y,w,h = detection[0]
                    cx= x+w/2
                    cy= y+h/2
                    centroids.append((cx,cy))
                    frame_inidices.append(frame_count) 
        cv2.imshow("Mask", mask)
        cv2.imshow("ROI",roi)
        # cv2.imshow("ROI_1", roi_1)
        cv2.imshow("Window", frame)
        # cv2.imwrite(filename,frame)
        out.write(frame)
        c=c+1
        frame_count+=1
        if cv2.waitKey(1000) & 0xff==ord("z"):
            break


video.release()
out.release()
cv2.destroyAllWindows()

velocities=[]
time=[]
projectile_sizes=[]
# Compute velocity between consecutive centroids
for i in range(1, len(centroids)):
    cx1, cy1 = centroids[i-1]
    cx2, cy2 = centroids[i]
    dx_px = cx2 - cx1
    dy_px = cy2 - cy1
    dx_m = dx_px * meters_per_pixel_x
    dy_m = dy_px * meters_per_pixel_y
    distance = math.hypot(dx_m, dy_m)
    dt = (frame_inidices[i]-frame_inidices[i-1])/ fps                                                             # consecutive frames
    if dt>0:
        velocity = distance / dt
        velocities.append(velocity)

        time.append(frame_inidices[i]/fps)                              #For equal interval of time, earlier it was (dt*i)
        print(f"Frame {frame_inidices[i-1]} -> {frame_inidices[i]}: "f"dx={dx_m:.4f} m, dy={dy_m:.4f} m, "f"distance={distance:.4f} m, velocity of Projectile ={velocity:.2f} m/s")

# # Converting the velocity data into a csv file for the further use:

# with open ("Velocity profile data.txt","w", newline="") as f:
#     writer= csv.writer(f)
#     # ourlist= [[dx_m],[dy_m],[distance],[velocity]]z
#     ourlist= [[velocity]]
#     writer.writerows(ourlist)


# Velocity Vs the Time for the travel of the projectile.
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))  # Wider figure
plt.plot(time, velocities, marker='o', linestyle='-', color='royalblue',markersize=6, linewidth=2, markerfacecolor='orange', markeredgecolor='black')

plt.xlabel("Time (s)", fontsize=12, fontweight='bold')
plt.ylabel("Velocity (m/s)", fontsize=12, fontweight='bold')
plt.title("Velocity vs Time", fontsize=14, fontweight='bold')

plt.grid(True, linestyle='--', alpha=0.6)
plt.tick_params(axis='both', which='major', labelsize=10)
plt.tight_layout()

plt.show()